









































































































































































































































































































































































































































# 目录
1. [商品数据处理](#section1)
 * 1.1 [商品数据的预处理](#section1_1)
 * 1.2 [读取评论、商品数据](#section1_2)
 * 1.3 [商品数据的选择](#section1_3)
 * 1.4 [规范商品数据](#section1_4)
 * 1.5 [结合用户行为数据](#section1_5)
2. [评论数据的推荐系统](#section2)
 * 2.1 [基于ALS（交替最小二乘法）的协同过滤推荐](#section2_1)  

# 1. 商品数据处理 <a class="anchor" id="section1"></a>

## 1.1 商品数据的预处理 <a class="anchor" id="section1_1"></a>

### 1.1.1 读取商品数据

In [ ]:
#数据处理
# 使用isin()方法截取数据
filtered_product_data = product[product['item_id'].isin(r_review['item_id'])]
# 删除rater_uid字段中的'u'
product['brand_id'] = product['brand_id'].str.replace('b', '')
# 删除rater_uid字段中的'u'
product['seller_id'] = product['seller_id'].str.replace('s', '')
# 对item_id列进行升序排序
sorted_product_data = filtered_product_data.sort_values(by='item_id', ascending=True)
#规范索引
sorted_product_data.reset_index(drop=True, inplace=True)
# 合并两个DataFrame
merged_df = product.join(spark_df, 'item_id', 'inner')

In [29]:
import pandas as pd

chunk_size = 500000 # 每一块的行数
chunks = []

for chunk in pd.read_csv('D:\\Datas\\tianchi_2014001_rec_tmall_product\\tianchi_2014001_rec_tmall_product.txt', sep="", chunksize=chunk_size):
    # 处理每个chunk的数据
    chunks.append(chunk)
    
# 合并chunk回一个大的DataFrame（如果需要的话）
product = pd.concat(chunks, axis=0)

In [30]:
import pandas as pd
from IPython.display import display, HTML

# 定义列名列表
column_names = ['item_id', 'title', 'pict_url','category', 'brand_id', 'seller_id']

# 设置DataFrame列名
product.columns = column_names

# 选择前10行数据
product.head(5)

,item_id,title,pict_url,category,brand_id,seller_id
0,5674056,孕妇装 秋装 时尚 韩版 泡泡袖 小 翻领 孕妇衬衫 孕妇上衣,http://img01.taobaocdn.com/bao/uploaded/i4/164...,16-2787,b26844,s78718
1,3095086,孕妇装 秋装 新款 孕妇裙 毛衣 线衫 孕妇上衣 假两件 时尚 长袖连衣裙,http://img01.taobaocdn.com/bao/uploaded/i1/135...,16-2787,b22178,s48736
2,3305168,2013 孕妇夏装 新款 孕妇装上衣 孕妇裙 夏装 时尚 韩版孕妇连衣裙,http://img01.taobaocdn.com/bao/uploaded/i2/135...,16-2787,b22178,s48736
3,7150118,孕妇装 秋装 新款 韩版 时尚 七分袖孕妇裙 休闲 孕妇上衣 棉麻 春秋 文艺,http://img01.taobaocdn.com/bao/uploaded/i4/135...,16-2787,b22178,s48736
4,6407723,2013 新款 孕妇 秋装连衣裙 时尚 孕妇裙子 韩版 夏装 春秋 冬装 孕妇装上衣,http://img01.taobaocdn.com/bao/uploaded/i4/135...,16-2787,b22178,s48736


### 1.1.2 规范数据

In [31]:
# 删除rater_uid字段中的'u'
product['brand_id'] = product['brand_id'].str.replace('b', '')

# 显示处理后的DataFrame
product.head()

,item_id,title,pict_url,category,brand_id,seller_id
0,5674056,孕妇装 秋装 时尚 韩版 泡泡袖 小 翻领 孕妇衬衫 孕妇上衣,http://img01.taobaocdn.com/bao/uploaded/i4/164...,16-2787,26844,s78718
1,3095086,孕妇装 秋装 新款 孕妇裙 毛衣 线衫 孕妇上衣 假两件 时尚 长袖连衣裙,http://img01.taobaocdn.com/bao/uploaded/i1/135...,16-2787,22178,s48736
2,3305168,2013 孕妇夏装 新款 孕妇装上衣 孕妇裙 夏装 时尚 韩版孕妇连衣裙,http://img01.taobaocdn.com/bao/uploaded/i2/135...,16-2787,22178,s48736
3,7150118,孕妇装 秋装 新款 韩版 时尚 七分袖孕妇裙 休闲 孕妇上衣 棉麻 春秋 文艺,http://img01.taobaocdn.com/bao/uploaded/i4/135...,16-2787,22178,s48736
4,6407723,2013 新款 孕妇 秋装连衣裙 时尚 孕妇裙子 韩版 夏装 春秋 冬装 孕妇装上衣,http://img01.taobaocdn.com/bao/uploaded/i4/135...,16-2787,22178,s48736


In [32]:
# 删除rater_uid字段中的'u'
product['seller_id'] = product['seller_id'].str.replace('s', '')

# 显示处理后的DataFrame
product.head()

,item_id,title,pict_url,category,brand_id,seller_id
0,5674056,孕妇装 秋装 时尚 韩版 泡泡袖 小 翻领 孕妇衬衫 孕妇上衣,http://img01.taobaocdn.com/bao/uploaded/i4/164...,16-2787,26844,78718
1,3095086,孕妇装 秋装 新款 孕妇裙 毛衣 线衫 孕妇上衣 假两件 时尚 长袖连衣裙,http://img01.taobaocdn.com/bao/uploaded/i1/135...,16-2787,22178,48736
2,3305168,2013 孕妇夏装 新款 孕妇装上衣 孕妇裙 夏装 时尚 韩版孕妇连衣裙,http://img01.taobaocdn.com/bao/uploaded/i2/135...,16-2787,22178,48736
3,7150118,孕妇装 秋装 新款 韩版 时尚 七分袖孕妇裙 休闲 孕妇上衣 棉麻 春秋 文艺,http://img01.taobaocdn.com/bao/uploaded/i4/135...,16-2787,22178,48736
4,6407723,2013 新款 孕妇 秋装连衣裙 时尚 孕妇裙子 韩版 夏装 春秋 冬装 孕妇装上衣,http://img01.taobaocdn.com/bao/uploaded/i4/135...,16-2787,22178,48736


### 1.1.3 保存并上传数据

In [33]:
#获保存处理后的商品数据
product.to_csv('D:\\Datas\\datas\\product.csv', index=False)

上传HDFS

<img src="images/改进/商品数据/hdfs新建文件夹.png" width="800" height="500">

<img src="images/改进/商品数据/上传至hdfs.png" width="800" height="500">

## 1.2 读取评论、商品数据 <a class="anchor" id="section1_2"></a>

### 1.2.1 读取评论数据

In [39]:
import pandas as pd


chunk_size = 500000 # 每一块的行数
chunks = []

for chunk in pd.read_csv('D:\\Datas\\datas\\r_review.csv', chunksize=chunk_size):
    # 处理每个chunk的数据
    chunks.append(chunk)
r_review = pd.concat(chunks, axis=0)
r_review.head(10)

,item_id,rater_uid,feedback,gmt_create,create_at,score,Result,score_class
0,163,6990067,很 甜 ， 个 大 。 谢谢 卖家 的 小 礼物 。,2013-06-28 10:58:01,2013-06-28,0.950070,好评,5
1,163,7203411,肉 香甜 ， 孩子 老人 都 喜欢 吃 ， 吃 完了 会 再 来,2013-06-27 23:59:36,2013-06-27,0.977025,好评,5
2,163,10604838,口感 不错 ， 个 大 ， 很 甜 哦 。 不错 ， 很 满意,2013-06-28 19:14:15,2013-06-28,0.999531,好评,5
3,163,9260044,很 好吃 ， 我 觉得 买 枣 都 可以 在 这家 买 了 ！,2013-07-09 13:11:03,2013-07-09,0.848073,好评,5
4,163,1684848,红枣 很 好吃 很 甜 ， 很 香 ， 很 好吃 服务 态度 也 很 好,2013-07-01 09:36:35,2013-07-01,0.997111,好评,5
5,163,8741365,不好意思 ， 出差 了 几 天 ， 昨天 才 回来 ， 同事 帮 签收 的 ， 这 红枣 真...,2013-06-29 09:13:13,2013-06-29,0.079785,差评,1
6,163,13180153,大枣 很 大 颗 很 甜 ， 家人 都 很 喜欢 吃 ， 吃 完了 再 来 买,2013-06-27 10:30:45,2013-06-27,0.981477,好评,5
7,163,3435601,实在 不好意思 ， 评价 晚 了 ， 大枣 都 吃 完了 ， 哈哈 很 好 ， 还 会 ...,2013-06-27 08:55:07,2013-06-27,0.422975,较差,3
8,183,6062783,不是说 是 均 码 么 长 了 很 多 呢,2013-08-07 12:15:21,2013-08-07,0.342111,较差,2
9,183,12885089,跟 想象 当中 的 一样 ， 非常 满意 . . ！,2013-06-14 09:20:24,2013-06-14,0.865261,好评,5


### 1.2.2 读取商品数据

In [40]:
import pandas as pd


chunk_size = 500000 # 每一块的行数
chunks = []

for chunk in pd.read_csv('D:/Datas/datas/product.csv', chunksize=chunk_size):
    # 处理每个chunk的数据
    chunks.append(chunk)
product = pd.concat(chunks, axis=0)
product.head(10)

,item_id,title,pict_url,category,brand_id,seller_id
0,5674056,孕妇装 秋装 时尚 韩版 泡泡袖 小 翻领 孕妇衬衫 孕妇上衣,http://img01.taobaocdn.com/bao/uploaded/i4/164...,16-2787,26844.0,78718
1,3095086,孕妇装 秋装 新款 孕妇裙 毛衣 线衫 孕妇上衣 假两件 时尚 长袖连衣裙,http://img01.taobaocdn.com/bao/uploaded/i1/135...,16-2787,22178.0,48736
2,3305168,2013 孕妇夏装 新款 孕妇装上衣 孕妇裙 夏装 时尚 韩版孕妇连衣裙,http://img01.taobaocdn.com/bao/uploaded/i2/135...,16-2787,22178.0,48736
3,7150118,孕妇装 秋装 新款 韩版 时尚 七分袖孕妇裙 休闲 孕妇上衣 棉麻 春秋 文艺,http://img01.taobaocdn.com/bao/uploaded/i4/135...,16-2787,22178.0,48736
4,6407723,2013 新款 孕妇 秋装连衣裙 时尚 孕妇裙子 韩版 夏装 春秋 冬装 孕妇装上衣,http://img01.taobaocdn.com/bao/uploaded/i4/135...,16-2787,22178.0,48736
5,8074944,孕妇装 春装 时尚 韩版 孕妇连衣裙 春 大码 孕妇裙子 新款 ...,http://img01.taobaocdn.com/bao/uploaded/i4/T1G...,16-2787,22178.0,48736
6,1165081,孕妇装 夏装 时尚新款 孕妇裙 韩版 孕妇 牛仔裙 连衣裙 纯手工 制作,http://img01.taobaocdn.com/bao/uploaded/i4/135...,16-2787,22178.0,48736
7,3888256,2013 秋装 孕妇装 新款 时尚 韩版孕妇裙 气质 雪纺孕妇连衣裙 子,http://img01.taobaocdn.com/bao/uploaded/i4/163...,16-2787,46543.0,39804
8,3034983,孕妇装 夏装 时尚 新款孕妇裙 韩国 孕妇装 气质 韩版孕妇连衣裙 子 夏,http://img01.taobaocdn.com/bao/uploaded/i2/163...,16-2787,46543.0,39804
9,294266,2013 秋装 孕妇装 新款孕妇裙 韩国 雪纺 孕妇装 韩版孕妇连衣裙 子,http://img01.taobaocdn.com/bao/uploaded/i2/163...,16-2787,46543.0,39804


## 1.3 商品数据的选择 <a class="anchor" id="section1_3"></a>

In [41]:
# 使用isin()方法
filtered_product_data = product[product['item_id'].isin(r_review['item_id'])]

filtered_product_data.head(10)  # 查看筛选后的商品数据前10行

,item_id,title,pict_url,category,brand_id,seller_id
6,1165081,孕妇装 夏装 时尚新款 孕妇裙 韩版 孕妇 牛仔裙 连衣裙 纯手工 制作,http://img01.taobaocdn.com/bao/uploaded/i4/135...,16-2787,22178.0,48736
9,294266,2013 秋装 孕妇装 新款孕妇裙 韩国 雪纺 孕妇装 韩版孕妇连衣裙 子,http://img01.taobaocdn.com/bao/uploaded/i2/163...,16-2787,46543.0,39804
13,5235270,2013 孕妇装 秋装新品 时尚 韩版 高端 韩国 雪纺上衣 孕妇 ...,http://img01.taobaocdn.com/bao/uploaded/i1/163...,16-2787,46543.0,39804
16,6134459,2013 夏装 新款 孕妇连衣裙 牛仔 碎花 拼接 时尚孕妇裙 韩版 显 瘦,http://img01.taobaocdn.com/bao/uploaded/i2/163...,16-2787,46543.0,39804
21,5497459,孕妇装 夏装 时尚 新款孕妇裙 韩国 孕妇装 韩版孕妇连衣裙 子 宽松,http://img01.taobaocdn.com/bao/uploaded/i4/163...,16-2787,46543.0,39804
22,2054303,孕妇装 夏装 时尚 新款 雪纺孕妇裙 孕妇装 气质 韩版孕妇连衣裙 子 夏,http://img01.taobaocdn.com/bao/uploaded/i3/163...,16-2787,46543.0,39804
55,7954188,女之御 孕妇裙 孕妇连衣裙 纽扣 双色 拼接 新款 孕妇装 12061,http://img01.taobaocdn.com/bao/uploaded/i2/T1t...,16-2787,34503.0,9712
75,4642698,孕妇装 孕妇裙 孕妇连衣裙 哺乳裙 夏款 大码 喂奶裙,http://img01.taobaocdn.com/bao/uploaded/i4/T1c...,16-2787,23309.0,43446
123,5720727,2013 新款 孕妇装 秋装连衣裙 时尚 孕妇裙子 孕妇 长袖上衣 秋冬装 包邮,http://img01.taobaocdn.com/bao/uploaded/i3/199...,16-2787,80779.0,38712
129,3321968,母亲节 新款 秋装 孕妇装 韩版 两件套连衣裙 条纹 哺乳衣 哺乳裙 月子 服,http://img01.taobaocdn.com/bao/uploaded/i1/199...,16-2787,80779.0,38712


## 1.4 规范商品数据 <a class="anchor" id="section1_4"></a>

In [43]:
# 对item_id列进行升序排序
sorted_product_data = filtered_product_data.sort_values(by='item_id', ascending=True)

# 打印排序后的结果
sorted_product_data.head(10)

,item_id,title,pict_url,category,brand_id,seller_id
6057804,163,【 大枣 】 熊猫 伯伯 新疆 和田 枣 五星 新疆 和田 红枣 大 玉枣 500 ...,http://img01.taobaocdn.com/bao/uploaded/i1/160...,9-1037,19063.0,49065
8013463,183,云柔 公主 高跟鱼嘴单鞋 豹纹 新款时尚 粗跟女鞋 鱼嘴 鞋子 女,http://img01.taobaocdn.com/bao/uploaded/i1/130...,90-4476,57252.0,34478
3193256,320,迪士尼 Disney 米奇 家族 保龄球 套装 儿童玩具 系列 ADJY 36126,http://img01.taobaocdn.com/bao/uploaded/i1/166...,74-2303,35189.0,63241
4181449,331,纯天然 粉 粉 食品 可可粉 340 g 原味 代 餐 巧克力 粉 海风堂 ...,http://img01.taobaocdn.com/bao/uploaded/i4/161...,37-3891,32755.0,4862
2186768,343,山东 东阿 产 百年 堂 阿胶 参 杞 颗粒 冲剂 500 g 买 3 送 1,http://img01.taobaocdn.com/bao/uploaded/i3/159...,76-5100,41809.0,71245
7441324,441,女单鞋 小辣椒 秋季 新秀 真皮 蝴蝶结 祼 黑 拼色 浅口 圆头 粗跟 ol 通勤 女鞋,http://img01.taobaocdn.com/bao/uploaded/i4/102...,90-4476,63330.0,55992
4910651,652,draconite 正品 潮 男式 休闲 帆布 纯色 单肩斜挎包 时尚 旅行运动包 147,http://img01.taobaocdn.com/bao/uploaded/i1/148...,48-6503,30994.0,4691
395914,670,适用 铁将军 折叠钥匙 改装 后 加 装 防盗器 匹配 学习型遥控器 汽车钥匙,http://img01.taobaocdn.com/bao/uploaded/i2/177...,84-3687,22779.0,71770
4463454,671,【 实 为 Coolpad / 酷派 8750 】 炫影 SII 5.5 大屏 Cool...,http://img01.taobaocdn.com/bao/uploaded/i1/119...,1-584,16108.0,20526
6210743,683,韩版 透气 时尚凉鞋 厚底 包头 平底鞋 女 松糕鞋 凉鞋 潮 2013 春 夏季,http://img01.taobaocdn.com/bao/uploaded/i3/120...,90-7241,87865.0,45418


In [46]:
#规范索引
sorted_product_data.reset_index(drop=True, inplace=True)
sorted_product_data.head(10)

,item_id,title,pict_url,category,brand_id,seller_id
0,163,【 大枣 】 熊猫 伯伯 新疆 和田 枣 五星 新疆 和田 红枣 大 玉枣 500 ...,http://img01.taobaocdn.com/bao/uploaded/i1/160...,9-1037,19063.0,49065
1,183,云柔 公主 高跟鱼嘴单鞋 豹纹 新款时尚 粗跟女鞋 鱼嘴 鞋子 女,http://img01.taobaocdn.com/bao/uploaded/i1/130...,90-4476,57252.0,34478
2,320,迪士尼 Disney 米奇 家族 保龄球 套装 儿童玩具 系列 ADJY 36126,http://img01.taobaocdn.com/bao/uploaded/i1/166...,74-2303,35189.0,63241
3,331,纯天然 粉 粉 食品 可可粉 340 g 原味 代 餐 巧克力 粉 海风堂 ...,http://img01.taobaocdn.com/bao/uploaded/i4/161...,37-3891,32755.0,4862
4,343,山东 东阿 产 百年 堂 阿胶 参 杞 颗粒 冲剂 500 g 买 3 送 1,http://img01.taobaocdn.com/bao/uploaded/i3/159...,76-5100,41809.0,71245
5,441,女单鞋 小辣椒 秋季 新秀 真皮 蝴蝶结 祼 黑 拼色 浅口 圆头 粗跟 ol 通勤 女鞋,http://img01.taobaocdn.com/bao/uploaded/i4/102...,90-4476,63330.0,55992
6,652,draconite 正品 潮 男式 休闲 帆布 纯色 单肩斜挎包 时尚 旅行运动包 147,http://img01.taobaocdn.com/bao/uploaded/i1/148...,48-6503,30994.0,4691
7,670,适用 铁将军 折叠钥匙 改装 后 加 装 防盗器 匹配 学习型遥控器 汽车钥匙,http://img01.taobaocdn.com/bao/uploaded/i2/177...,84-3687,22779.0,71770
8,671,【 实 为 Coolpad / 酷派 8750 】 炫影 SII 5.5 大屏 Cool...,http://img01.taobaocdn.com/bao/uploaded/i1/119...,1-584,16108.0,20526
9,683,韩版 透气 时尚凉鞋 厚底 包头 平底鞋 女 松糕鞋 凉鞋 潮 2013 春 夏季,http://img01.taobaocdn.com/bao/uploaded/i3/120...,90-7241,87865.0,45418


In [47]:
sorted_product_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108389 entries, 0 to 108388
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   item_id    108389 non-null  int64  
 1   title      108389 non-null  object 
 2   pict_url   108389 non-null  object 
 3   category   108389 non-null  object 
 4   brand_id   105122 non-null  float64
 5   seller_id  108389 non-null  int64  
dtypes: float64(1), int64(2), object(3)
memory usage: 5.0+ MB


In [48]:
#获得商品数据
sorted_product_data.to_csv('D:\\Datas\\datas\\product_1.csv', index=False)

## 1.5 结合用户行为数据 <a class="anchor" id="section1_5"></a>

In [1]:
#运行时候使用Spark不会报错
import findspark
findspark.init()

In [2]:
# spark配置信息
from pyspark import SparkConf
from pyspark.sql import SparkSession

SPARK_APP_NAME = "product"

conf = SparkConf()    # 创建spark config对象
config = (
    ("spark.app.name", SPARK_APP_NAME),    # 设置启动的spark的app名称，没有提供，将随机产生一个名称
    ("spark.executor.memory", "12g"),    # 设置该app启动时占用的内存用量，默认1g
    ("spark.executor.cores", "4"),    # 设置spark executor使用的CPU核心数
)
conf.setAll(config)
## 1.1 商品数据的预处理 <a class="anchor" id="section1_1"></a>
# 利用config对象，创建spark session
spark = SparkSession.builder.config(conf=conf).getOrCreate()

###  1.5.1 读取商品数据

In [3]:
# 从hdfs加载CSV文件为DataFrame
product = spark.read.csv("hdfs://localhost:9000/E_commerce_platform/product/product_1.csv", header=True)
product.show()    # 查看dataframe，默认显示前20条
# 大致查看一下数据类型
product.printSchema()    # 打印当前dataframe的结构

+-------+--------------------------------+--------------------+--------+--------+---------+
|item_id|                           title|            pict_url|category|brand_id|seller_id|
+-------+--------------------------------+--------------------+--------+--------+---------+
|    163|  【 大枣 】 熊猫 伯伯   新疆...|http://img01.taob...|  9-1037| 19063.0|    49065|
|    183|  云柔 公主   高跟鱼嘴单鞋   ...|http://img01.taob...| 90-4476| 57252.0|    34478|
|    320|      迪士尼 Disney   米奇 家...|http://img01.taob...| 74-2303| 35189.0|    63241|
|    331|  纯天然 粉 粉 食品   可可粉 ...|http://img01.taob...| 37-3891| 32755.0|     4862|
|    343| 山东 东阿 产 百年 堂 阿胶 参...|http://img01.taob...| 76-5100| 41809.0|    71245|
|    441|女单鞋 小辣椒 秋季 新秀 真皮 ...|http://img01.taob...| 90-4476| 63330.0|    55992|
|    652|       draconite 正品 潮 男式...|http://img01.taob...| 48-6503| 30994.0|     4691|
|    670| 适用 铁将军 折叠钥匙 改装   ...|http://img01.taob...| 84-3687| 22779.0|    71770|
|    671|        【 实 为 Coolpad / 酷...|http://img01.taob...|   1

### 1.5.2 设置结构

In [4]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, FloatType, TimestampType
from pyspark.sql.functions import unix_timestamp
from pyspark.sql.functions import from_unixtime
# 构建结构对象
schema = StructType([
    StructField("item_id", IntegerType()),
    StructField("title", StringType()),
    StructField("pict_url", StringType()),
    StructField("category", StringType()),
    StructField("brand_id", StringType()),
    StructField("seller_id", IntegerType()),
])
# 从hdfs加载数据为dataframe，并设置结构
product = spark.read.csv("hdfs://localhost:9000/E_commerce_platform/product/product_1.csv", header=True, schema=schema)
product.show()
# review_df.count() 
product.printSchema() 

+-------+--------------------------------+--------------------+--------+--------+---------+
|item_id|                           title|            pict_url|category|brand_id|seller_id|
+-------+--------------------------------+--------------------+--------+--------+---------+
|    163|  【 大枣 】 熊猫 伯伯   新疆...|http://img01.taob...|  9-1037| 19063.0|    49065|
|    183|  云柔 公主   高跟鱼嘴单鞋   ...|http://img01.taob...| 90-4476| 57252.0|    34478|
|    320|      迪士尼 Disney   米奇 家...|http://img01.taob...| 74-2303| 35189.0|    63241|
|    331|  纯天然 粉 粉 食品   可可粉 ...|http://img01.taob...| 37-3891| 32755.0|     4862|
|    343| 山东 东阿 产 百年 堂 阿胶 参...|http://img01.taob...| 76-5100| 41809.0|    71245|
|    441|女单鞋 小辣椒 秋季 新秀 真皮 ...|http://img01.taob...| 90-4476| 63330.0|    55992|
|    652|       draconite 正品 潮 男式...|http://img01.taob...| 48-6503| 30994.0|     4691|
|    670| 适用 铁将军 折叠钥匙 改装   ...|http://img01.taob...| 84-3687| 22779.0|    71770|
|    671|        【 实 为 Coolpad / 酷...|http://img01.taob...|   1

### 1.5.3 读取关于商品的操作数据

In [20]:
import pandas as pd

# 读取CSV文件为Pandas DataFrame
action = pd.read_csv("D:/Datas/datas/filtered_data.csv")
action.head(10)

,item_id,user_id,action,vtime
0,6289870,u3427707,click,2013-08-30 09:27:15
1,4120848,u3427707,click,2013-05-22 16:54:34
2,4120848,u3427707,click,2013-05-24 12:27:31
3,4690166,u3427707,click,2013-05-22 16:55:10
4,4690166,u3427707,click,2013-05-24 12:27:39
5,4690166,u3427707,click,2013-05-22 16:54:34
6,6301705,u3427707,click,2013-07-29 16:43:02
7,6301705,u3427707,click,2013-07-29 16:54:39
8,6025858,u3427707,click,2013-06-04 12:35:32
9,1528104,u5853382,click,2013-09-16 13:10:44


### 1.5.4 分组

In [21]:
df = pd.DataFrame(action)

# 使用groupby和agg进行分组并统计数量
result = df.groupby(['item_id', 'action']).agg(count=('user_id', 'count')).reset_index()
result.head(10)

,item_id,action,count
0,163,alipay,1
1,163,cart,4
2,163,click,113
3,163,collect,1
4,183,alipay,6
5,183,cart,17
6,183,click,666
7,183,collect,25
8,320,alipay,6
9,320,cart,10


### 1.5.5 拆分

In [22]:
#方便查找
import pandas as pd

df = pd.DataFrame(result)

# 使用pivot函数将action字段拆分为四个字段
df_pivot = df.pivot(index='item_id', columns='action', values='count').reset_index()
df_pivot = df_pivot.rename_axis(None, axis=1)

# 填充缺失值（NaN）并重命名列名
df_pivot = df_pivot.fillna(0)
df_pivot = df_pivot.astype({'alipay': int, 'cart': int, 'click': int, 'collect': int})
df_pivot.head()

,item_id,alipay,cart,click,collect
0,163,1,4,113,1
1,183,6,17,666,25
2,320,6,10,85,0
3,331,2,10,155,1
4,343,4,1,246,4


In [23]:
# 将Pandas DataFrame转换为Spark DataFrame
spark_df = spark.createDataFrame(df_pivot)

# 打印Spark DataFrame的schema
spark_df.printSchema()

# 展示前几行数据
spark_df.show(10)

root
 |-- item_id: long (nullable = true)
 |-- alipay: long (nullable = true)
 |-- cart: long (nullable = true)
 |-- click: long (nullable = true)
 |-- collect: long (nullable = true)

+-------+------+----+-----+-------+
|item_id|alipay|cart|click|collect|
+-------+------+----+-----+-------+
|    163|     1|   4|  113|      1|
|    183|     6|  17|  666|     25|
|    320|     6|  10|   85|      0|
|    331|     2|  10|  155|      1|
|    343|     4|   1|  246|      4|
|    441|     7|   9|  280|     17|
|    652|     3|   5|  341|      6|
|    670|     3|   3|  293|      3|
|    671|     3|  11| 2268|     25|
|    683|     8|  33|  609|     33|
+-------+------+----+-----+-------+
only showing top 10 rows



### 1.5.6 合并

In [24]:
# 合并两个DataFrame
merged_df = product.join(spark_df, 'item_id', 'inner')

# 展示合并后的数据
merged_df.show(10)

+-------+---------------------------------+--------------------+--------+--------+---------+------+----+-----+-------+
|item_id|                            title|            pict_url|category|brand_id|seller_id|alipay|cart|click|collect|
+-------+---------------------------------+--------------------+--------+--------+---------+------+----+-----+-------+
|   9945|   服装 工业 打 板 技术 全 编 ...|http://img01.taob...| 35-1014|    null|    33061|     2|   1|   37|      4|
|  11190|修正 大豆 提取物 固体饮料 女性...|http://img01.taob...| 24-2601| 72491.0|    38150|     9|   5|  100|      4|
|  15322|    青 见 飘雪     蒙山 茉莉花...|http://img01.taob...| 37-5225|  9952.0|    35992|     0|   0|   24|      0|
| 101519|      爱 桥   2013 秋装 新款牛...|http://img01.taob...|  10-802|  7092.0|     1099|     4|  13|  164|      6|
| 118628|  环形 扶手 靠背椅   实木家具 ...|http://img01.taob...| 54-3799| 22835.0|    14148|     5|   7|   64|      2|
| 126753|        7.8 - 7.14 上海 世博园...|http://img01.taob...| 66-2586|    null|    26403|    11|   0

In [25]:
# 根据 item_id 进行排序
sorted_df = merged_df.orderBy('item_id')

# 展示排序后的数据
sorted_df.show(10)

+-------+--------------------------------+--------------------+--------+--------+---------+------+----+-----+-------+
|item_id|                           title|            pict_url|category|brand_id|seller_id|alipay|cart|click|collect|
+-------+--------------------------------+--------------------+--------+--------+---------+------+----+-----+-------+
|    163|  【 大枣 】 熊猫 伯伯   新疆...|http://img01.taob...|  9-1037| 19063.0|    49065|     1|   4|  113|      1|
|    183|  云柔 公主   高跟鱼嘴单鞋   ...|http://img01.taob...| 90-4476| 57252.0|    34478|     6|  17|  666|     25|
|    320|      迪士尼 Disney   米奇 家...|http://img01.taob...| 74-2303| 35189.0|    63241|     6|  10|   85|      0|
|    331|  纯天然 粉 粉 食品   可可粉 ...|http://img01.taob...| 37-3891| 32755.0|     4862|     2|  10|  155|      1|
|    343| 山东 东阿 产 百年 堂 阿胶 参...|http://img01.taob...| 76-5100| 41809.0|    71245|     4|   1|  246|      4|
|    441|女单鞋 小辣椒 秋季 新秀 真皮 ...|http://img01.taob...| 90-4476| 63330.0|    55992|     7|   9|  280|     1

### 1.5.7 保存数据

In [26]:
# 将DataFrame转换为Pandas DataFrame
pandas_df = sorted_df.toPandas()
# 保存为 CSV 文件
pandas_df.to_csv('D:\\Datas\\datas\\product_3.csv', index=False)

# 2. 评论数据的推荐系统  <a class="anchor" id="section2"></a>

## 2.1 基于ALS（交替最小二乘法）的协同过滤推荐  <a class="anchor" id="section2_1"></a>

>使用Spark来实现基于ALS（交替最小二乘法）的协同过滤推荐用于在用户-物品评分数据上进行推荐。

ALS(maxIter=5, regParam=0.01, userCol="rater_uid", itemCol="item_id", ratingCol="score_class",
          coldStartStrategy="drop")
- 参数说明如下：
* maxIter：迭代次数，指定算法运行的最大迭代次数。
* regParam：正则化参数，用于控制模型的正则化程度。较大的值可以减少过拟合的风险。
* userCol：指定包含用户ID的列名。
* itemCol：指定包含物品ID的列名。
* ratingCol：指定包含评分的列名。
* coldStartStrategy：冷启动策略，用于处理在训练期间遇到的新用户或新物品。"drop"表示在训练过程中删除具有冷启动的行。

In [2]:
import findspark
findspark.init()

### 2.1.1 读取数据

In [3]:
from pyspark.sql import SparkSession
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.sql import Row

# 创建 Spark 会话
# spark配置信息
from pyspark import SparkConf
from pyspark.sql import SparkSession

SPARK_APP_NAME = "r_review"

conf = SparkConf()    # 创建spark config对象
config = (
    ("spark.app.name", SPARK_APP_NAME),    # 设置启动的spark的app名称，没有提供，将随机产生一个名称
    ("spark.executor.memory", "12g"),    # 设置该app启动时占用的内存用量，默认1g
    ("spark.executor.cores", "4"),    # 设置spark executor使用的CPU核心数
)
conf.setAll(config)
# 利用config对象，创建spark session
spark = SparkSession.builder.config(conf=conf).getOrCreate()

In [4]:
import pandas as pd

# action为大量数据的DataFrame
chunk_size = 100000  # 块大小
chunks = []

# 逐块处理数据
for chunk in pd.read_csv('D:\\Datas\\datas\\r_review.csv', chunksize=chunk_size):
    chunks.append(chunk)
review = pd.concat(chunks, axis=0)
review.head(10)

,item_id,rater_uid,feedback,gmt_create,create_at,score,Result,score_class
0,163,6990067,很 甜 ， 个 大 。 谢谢 卖家 的 小 礼物 。,2013-06-28 10:58:01,2013-06-28,0.950070,好评,5
1,163,7203411,肉 香甜 ， 孩子 老人 都 喜欢 吃 ， 吃 完了 会 再 来,2013-06-27 23:59:36,2013-06-27,0.977025,好评,5
2,163,10604838,口感 不错 ， 个 大 ， 很 甜 哦 。 不错 ， 很 满意,2013-06-28 19:14:15,2013-06-28,0.999531,好评,5
3,163,9260044,很 好吃 ， 我 觉得 买 枣 都 可以 在 这家 买 了 ！,2013-07-09 13:11:03,2013-07-09,0.848073,好评,5
4,163,1684848,红枣 很 好吃 很 甜 ， 很 香 ， 很 好吃 服务 态度 也 很 好,2013-07-01 09:36:35,2013-07-01,0.997111,好评,5
5,163,8741365,不好意思 ， 出差 了 几 天 ， 昨天 才 回来 ， 同事 帮 签收 的 ， 这 红枣 真...,2013-06-29 09:13:13,2013-06-29,0.079785,差评,1
6,163,13180153,大枣 很 大 颗 很 甜 ， 家人 都 很 喜欢 吃 ， 吃 完了 再 来 买,2013-06-27 10:30:45,2013-06-27,0.981477,好评,5
7,163,3435601,实在 不好意思 ， 评价 晚 了 ， 大枣 都 吃 完了 ， 哈哈 很 好 ， 还 会 ...,2013-06-27 08:55:07,2013-06-27,0.422975,较差,3
8,183,6062783,不是说 是 均 码 么 长 了 很 多 呢,2013-08-07 12:15:21,2013-08-07,0.342111,较差,2
9,183,12885089,跟 想象 当中 的 一样 ， 非常 满意 . . ！,2013-06-14 09:20:24,2013-06-14,0.865261,好评,5


In [5]:
ratings = spark.createDataFrame(review[["item_id", "rater_uid", "score_class"]])
ratings.show()

+-------+---------+-----------+
|item_id|rater_uid|score_class|
+-------+---------+-----------+
|    163|  6990067|          5|
|    163|  7203411|          5|
|    163| 10604838|          5|
|    163|  9260044|          5|
|    163|  1684848|          5|
|    163|  8741365|          1|
|    163| 13180153|          5|
|    163|  3435601|          3|
|    183|  6062783|          2|
|    183| 12885089|          5|
|    183|  2647879|          5|
|    183|  4113264|          5|
|    183|  1988451|          5|
|    183| 11637563|          5|
|    183|  1147032|          5|
|    183| 13952537|          5|
|    320|  5004497|          5|
|    320| 15070290|          2|
|    320| 11186270|          5|
|    320|  6836630|          2|
+-------+---------+-----------+
only showing top 20 rows



### 2.1.2 构建模型

In [6]:
als = ALS(maxIter=5, regParam=0.01, userCol="rater_uid", itemCol="item_id", ratingCol="score_class",
          coldStartStrategy="drop")
model = als.fit(ratings)

### 2.1.3 生成推荐结果

In [7]:
userRecs = model.recommendForAllUsers(5)  # 推荐每个用户前5个物品
userRecs.show(10)

+---------+--------------------+
|rater_uid|     recommendations|
+---------+--------------------+
|     1645|[[7689847, 5.4035...|
|     4519|[[1377463, 3.0957...|
|     4935|[[7021042, 5.2392...|
|     5518|[[3359728, 1.1379...|
|    10206|[[7814424, 4.9996...|
|    12799|[[773360, 7.13961...|
|    13623|[[496871, 1.12206...|
|    15619|[[7452417, 5.3210...|
|    16386|[[287828, 6.40402...|
|    17389|[[5274526, 5.3386...|
+---------+--------------------+
only showing top 10 rows



In [8]:
# 将DataFrame转换为Pandas DataFrame
pandas_df = userRecs.toPandas()

In [9]:
pandas_df.head(5)

,rater_uid,recommendations
0,1645,"[(7689847, 5.40358829498291), (5372890, 5.1547..."
1,4519,"[(1377463, 3.0957834720611572), (25892, 3.0043..."
2,4935,"[(7021042, 5.239245891571045), (54614, 5.16512..."
3,5518,"[(3359728, 1.137943983078003), (7858255, 1.065..."
4,10206,"[(7814424, 4.999618053436279), (2463324, 4.993..."


In [ ]:
# 查看DataFrame的大小
print(pandas_df.shape)

In [10]:
#保存
pandas_df.to_csv('D:\\Datas\\datas\\recommendations.csv', index=False)

### 2.1.4 关闭会话

In [11]:
spark.stop()

## 2.2 ALS 模型评估

### 2.2.1 加载数据

In [12]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql.functions import col

# 创建Spark会话
# spark配置信息
from pyspark import SparkConf
from pyspark.sql import SparkSession

SPARK_APP_NAME = "r_review"

conf = SparkConf()    # 创建spark config对象
config = (
    ("spark.app.name", SPARK_APP_NAME),    # 设置启动的spark的app名称，没有提供，将随机产生一个名称
    ("spark.executor.memory", "12g"),    # 设置该app启动时占用的内存用量，默认1g
    ("spark.executor.cores", "4"),    # 设置spark executor使用的CPU核心数
)
conf.setAll(config)
# 利用config对象，创建spark session
spark = SparkSession.builder.config(conf=conf).getOrCreate()

# 加载评分数据集
# action为大量数据的DataFrame
chunk_size = 100000  # 块大小
chunks = []

# 逐块处理数据
for chunk in pd.read_csv('D:\\Datas\\datas\\r_review.csv', chunksize=chunk_size):
    chunks.append(chunk)
review = pd.concat(chunks, axis=0)
ratings = spark.createDataFrame(review[["item_id", "rater_uid", "score_class"]])
ratings.show()

+-------+---------+-----------+
|item_id|rater_uid|score_class|
+-------+---------+-----------+
|    163|  6990067|          5|
|    163|  7203411|          5|
|    163| 10604838|          5|
|    163|  9260044|          5|
|    163|  1684848|          5|
|    163|  8741365|          1|
|    163| 13180153|          5|
|    163|  3435601|          3|
|    183|  6062783|          2|
|    183| 12885089|          5|
|    183|  2647879|          5|
|    183|  4113264|          5|
|    183|  1988451|          5|
|    183| 11637563|          5|
|    183|  1147032|          5|
|    183| 13952537|          5|
|    320|  5004497|          5|
|    320| 15070290|          2|
|    320| 11186270|          5|
|    320|  6836630|          2|
+-------+---------+-----------+
only showing top 20 rows



### 2.2.2 创建模型

In [13]:
# 数据集划分为训练集和测试集
(training, test) = ratings.randomSplit([0.8, 0.2])

# 创建ALS模型
als = ALS(maxIter=5, regParam=0.01, userCol="rater_uid", itemCol="item_id", ratingCol="score_class",
          coldStartStrategy="drop")

# 在训练集上拟合ALS模型
model = als.fit(training)

### 2.2.3 预测

In [14]:
# 在测试集上进行预测
predictions = model.transform(test)
predictions.show(10)

+-------+---------+-----------+----------+
|item_id|rater_uid|score_class|prediction|
+-------+---------+-----------+----------+
|  57380| 15010426|          5|0.41660404|
|  57380|  1960282|          5|0.27701366|
|  82529| 11488240|          3|0.49286205|
|  97419| 14092650|          1| 2.0664124|
| 115602| 14603190|          3| 0.8057704|
| 228725|  2018265|          5| 2.7622614|
| 239148|  8653511|          5| 1.8164016|
| 258032| 13544516|          5|  4.332679|
| 258032| 13327501|          5|  4.133185|
| 424170|  5146067|          5|  1.027491|
+-------+---------+-----------+----------+
only showing top 10 rows



### 2.2.4 RMSE 与 MAE

> 均方根误差（RMSE）和平均绝对误差（MAE）：这两个评估指标用于衡量预测评分与实际评分之间的差异。RMSE和MAE的值越小，表示模型的预测准确性越高。

In [15]:
# 定义评估器（使用均方根误差和平均绝对误差）
evaluator_rmse = RegressionEvaluator(metricName="rmse", labelCol="score_class", predictionCol="prediction")
evaluator_mae = RegressionEvaluator(metricName="mae", labelCol="score_class", predictionCol="prediction")

# 计算均方根误差
rmse = evaluator_rmse.evaluate(predictions)
print("均方根误差 (RMSE) = {:.4f}".format(rmse))

# 计算平均绝对误差
mae = evaluator_mae.evaluate(predictions)
print("平均绝对误差 (MAE) = {:.4f}".format(mae))

均方根误差 (RMSE) = 3.7027
平均绝对误差 (MAE) = 2.8578


1. RMSE:衡量模型预测值与实际观测值之间差异的指标。RMSE的值越小，表示模型的预测准确性越高。RMSE为3.7027，这意味着模型的预测值平均上下浮动约3.7027个单位。
2. MAE:衡量模型预测值与实际观测值之间绝对差异的指标。MAE的值越小，表示模型的预测准确性越高。MAE为2.8578，这意味着模型的平均预测误差约为2.8578个单位。
3. RMSE为3.7027和MAE为2.8578，可以说模型的预测性能有一定的改进空间，因为较小的值表示模型的预测误差相对较小。根据评估结果该模型的效果要劣于用用户行为做的模型

### 2.2.5 准确率和召回率

>准确率和召回率：这两个评估指标用于衡量推荐系统的精确性和覆盖率。准确率表示推荐列表中真正被用户喜欢的物品所占的比例，召回率表示真正被用户喜欢的物品在推荐列表中的比例。准确率和召回率的值都在0到1之间，越接近1表示推荐系统的性能越好。

In [16]:
from pyspark.sql.functions import col
from pyspark.sql.functions import collect_list, struct, explode
# 计算准确率和召回率
top_n = 5  # 推荐列表中的前n个物品
predictions_top_n = predictions.select("item_id", "rater_uid", "prediction") \
    .groupBy("rater_uid") \
    .agg(collect_list(struct("item_id", "prediction")).alias("recommendations"))

# 计算准确率和召回率
true_positive = predictions_top_n.join(test, ["rater_uid"]).selectExpr(
    "rater_uid", "item_id as true_item_id", "explode(recommendations.item_id) as recommended_item_id"
).where("recommended_item_id = true_item_id").count()

total_relevant = test.count()
total_recommended = predictions_top_n.select("rater_uid", explode("recommendations.item_id")).distinct().count()

precision = true_positive / total_recommended
recall = true_positive / total_relevant

print("准确率 = {:.4f}".format(precision))
print("召回率 = {:.4f}".format(recall))

准确率 = 1.0810
召回率 = 0.1720


In [17]:
# 关闭Spark会话
spark.stop()